# 00 · Setup and smoke test

Runs on a **CPU** runtime. Installs the repo, mounts Drive, runs the unit tests, and runs the adaptive lifecycle coordinator on synthetic data with planted collapse, so the agent system can be developed before any model outputs exist.

In [ ]:
# Clone (or update) the repo and install it. Re-run after a runtime reset.
import os, subprocess
REPO_DIR = "/content/mpac"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hurshprasad/Multi-Agent-Diagnosis-of-Multilingual-Pluralistic-Alignment.git $REPO_DIR
else:
    !git -C $REPO_DIR pull --ff-only
%cd $REPO_DIR
!pip install -q -e ".[dev]"
import sys
sys.path.insert(0, f"{REPO_DIR}/src")  # editable installs aren't always visible to a running kernel

In [ ]:
CONFIG = "configs/smoke.yaml"
# Mount Drive so generations, checkpoints and agent state survive runtime resets.
from google.colab import drive
drive.mount("/content/drive")
from mpac import load_config
from mpac.paths import resolve_root
cfg = load_config(CONFIG)
print("artifacts ->", resolve_root(cfg))

In [ ]:
!python -m pytest -q

## Adaptive lifecycle on synthetic data
Collapse is planted in German/honesty and Italian/pluralistic value. The coordinator should spawn specialists for exactly those cells.

In [ ]:
from mpac.config import load_config
from mpac.data.synthetic import make_synthetic
from mpac.agents.coordinator import LifecycleCoordinator
from mpac.analysis.metrics import detection_scores

cfg = load_config()
cfg["agents"].update(population_cap=12, lifecycle_cycles=5, batch_per_cycle=30)
targets, objectives = cfg["languages"]["targets"], cfg["objectives"]
planted = {("de", "honesty"): 0.9, ("it", "pluralistic_value"): 0.8}
items, gens, oracle = make_synthetic(targets, objectives, n_items=150, collapsed=planted)

coord = LifecycleCoordinator(cfg, items, gens, oracle)
summary = coord.run()
print("lifecycle ops:", summary["lifecycle_ops"])
print("population by cycle:", summary["population_by_cycle"])
print("alive:", *summary["final_alive"], sep="\n  ")
print(detection_scores(coord.all_evals, oracle))

In [ ]:
import pandas as pd
cmap = pd.DataFrame(summary["collapse_map"]).T
cmap.index.names = ["lang", "objective"]
cmap.sort_values("collapse_rate", ascending=False).round(3)